# SOC identity investigation lab
Reconstructed from the supplied walkthrough. Run all six code cells in order. Synthetic events and loopback-only response demonstration; no Azure, Jira or identity-provider integration. Outputs are intentionally empty until you execute.

In [ ]:
from datetime import datetime, timedelta, timezone
import json, time, threading, queue, secrets, hashlib, hmac
from pathlib import Path

BASE = datetime(2026, 9, 23, 9, 0, tzinfo=timezone.utc)


def event(i, seconds, result, user="alex", ip="192.0.2.10"):
    return dict(
        id=i,
        time=BASE + timedelta(seconds=seconds),
        user=user,
        ip=ip,
        app="DeliveryPortal",
        result=result,
    )


def detect(events):
    unique = {e["id"]: e for e in events}
    rows = sorted(unique.values(), key=lambda e: (e["time"], e["id"]))
    alerts = []
    for success in rows:
        if success["result"] != "Success":
            continue
        failures = [
            e
            for e in rows
            if e["result"] == "Failed"
            and all(e[k] == success[k] for k in ("user", "ip", "app"))
            and success["time"] - timedelta(minutes=5)
            <= e["time"]
            < success["time"]
        ]
        if len(failures) >= 3:
            alerts.append(
                dict(
                    alert_id="rule1:" + success["id"],
                    user=success["user"],
                    ip=success["ip"],
                    application=success["app"],
                    failures=len(failures),
                    success_time=success["time"].isoformat(),
                    evidence_ids=[e["id"] for e in failures]
                    + [success["id"]],
                    disposition="Needs analyst review",
                )
            )
    return alerts


pattern = [
    event("f1", 0, "Failed"),
    event("f2", 60, "Failed"),
    event("f3", 120, "Failed"),
    event("s1", 180, "Success"),
]
tests = {
    "Failures then success": (pattern, 1),








    "One typo then success": (
        [event("a", 0, "Failed"), event("b", 60, "Success")],
        0,
    ),
    "Failures without success": (pattern[:3], 0),
    "Success before failures": (
        [event("early", -60, "Success")] + pattern[:3],
        0,
    ),
    "Failures outside five minutes": (
        pattern[:3] + [event("late", 600, "Success")],
        0,
    ),
    "Duplicates must not inflate count": (
        [pattern[0]] * 3 + [pattern[3]],
        0,
    ),
    "Exactly five-minute boundary": (
        [
            event("a", 0, "Failed"),
            event("b", 60, "Failed"),
            event("c", 120, "Failed"),
            event("d", 300, "Success"),
        ],
        1,
    ),
    "Equal timestamps not proven before": (
        [
            event("a", 0, "Failed"),
            event("b", 0, "Failed"),
            event("c", 0, "Failed"),
            event("d", 0, "Success"),
        ],
        0,
    ),
    "Other IP must not be combined": (
        pattern[:3] + [event("other", 180, "Success", ip="198.51.100.20")],
        0,
    ),
    "Legitimate repeated typos, same pattern": (pattern, 1),
}
print("Detector ready. Next cell runs its tests.")


In [ ]:
for label, (rows, expected) in tests.items():
    actual = len(detect(rows))
    assert actual == expected, (label, actual, expected)
    print(f"PASS | {label}: {actual} alert(s)")
print("10 checks passed. A matching pattern still needs investigation.")


In [ ]:
incoming = queue.Queue()
emitted, seen, history = [], set(), []


def alert_worker():
    while True:
        item = incoming.get()
        try:
            if item is None:
                return
            history.append(item)
            for alert in detect(history):
                if alert["alert_id"] not in seen:
                    seen.add(alert["alert_id"])
                    emitted.append(alert)
                    print(
                        "AUTOMATIC ALERT:",
                        alert["user"],
                        "-",
                        alert["failures"],
                        "failures before success",
                    )
        finally:
            incoming.task_done()


worker = threading.Thread(target=alert_worker, daemon=True)
worker.start()
for row in pattern + [pattern[-1]]:
    incoming.put(row)
    time.sleep(0.15)
incoming.put(None)
worker.join(timeout=5)
assert not worker.is_alive(), "Worker did not stop"
assert len(emitted) == 1
Path("project1_alerts.jsonl").write_text(
    "\n".join(json.dumps(a) for a in emitted) + "\n"
)
print("PASS: one alert, repeated delivery suppressed, worker stopped.")
print(json.dumps(emitted[0], indent=2))


In [ ]:
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.request import Request, urlopen
from urllib.error import HTTPError

if "server" in globals():
    server.shutdown()
    server.server_close()
OLD_PASSWORD = "Lab-only-before-482!"
NEW_PASSWORD = "Lab-only-after-729!"
salt = secrets.token_bytes(16)


def password_hash(value):
    return hashlib.pbkdf2_hmac("sha256", value.encode(), salt, 100000)


account = {
    "user": "alex",
    "enabled": True,
    "password_hash": password_hash(OLD_PASSWORD),
}
sessions, audit = {}, []


def record(action, result):
    audit.append(
        {
            "time": datetime.now(timezone.utc).isoformat(),
            "action": action,
            "result": result,
        }
    )


class Handler(BaseHTTPRequestHandler):
    def log_message(self, *args):
        pass

    def reply(self, status, body):
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.end_headers()
        self.wfile.write(json.dumps(body).encode())

    def do_POST(self):
        if self.path != "/login":
            return self.reply(404, {"error": "unknown route"})
        data = json.loads(
            self.rfile.read(int(self.headers.get("Content-Length", 0)))
        )
        if not account["enabled"]:
            record("login", "blocked-disabled")
            return self.reply(403, {"error": "account disabled"})
        if data.get("user") != "alex" or not hmac.compare_digest(
            password_hash(data.get("password", "")),
            account["password_hash"],
        ):
            record("login", "bad-credentials")
            return self.reply(401, {"error": "bad credentials"})








        token = secrets.token_urlsafe(24)
        sessions[token] = "alex"
        record("login", "success")
        self.reply(200, {"token": token})

    def do_GET(self):
        if self.path != "/order":
            return self.reply(404, {"error": "unknown route"})
        token = self.headers.get("Authorization", "").removeprefix(
            "Bearer "
        )
        if token not in sessions:
            record("view-order", "rejected-session")
            return self.reply(401, {"error": "invalid session"})
        if not account["enabled"]:
            record("view-order", "blocked-disabled")
            return self.reply(403, {"error": "account disabled"})
        record("view-order", "success")
        self.reply(
            200,
            {"order": "TRAINING-1042", "contents": "fictional test order"},
        )


server = ThreadingHTTPServer(("127.0.0.1", 0), Handler)
threading.Thread(target=server.serve_forever, daemon=True).start()
base_url = "http://127.0.0.1:" + str(server.server_port)


def request(path, payload=None, token=None):
    headers = {"Content-Type": "application/json"}
    if token:
        headers["Authorization"] = "Bearer " + token
    req = Request(
        base_url + path,
        data=json.dumps(payload).encode() if payload else None,
        headers=headers,
    )
    try:
        with urlopen(req, timeout=3) as response:
            return response.status, json.loads(response.read())
    except HTTPError as err:
        return err.code, json.loads(err.read())


status, body = request("/login", {"user": "alex", "password": OLD_PASSWORD})
assert status == 200
old_token = body["token"]
assert request("/order", token=old_token)[0] == 200
print("BEFORE containment: login 200; protected order access 200.")


In [ ]:
sessions.clear()
record("admin-revoke-sessions", "completed")
account["enabled"] = False
record("admin-disable-account", "completed")
old_access = request("/order", token=old_token)[0]
fresh_login = request("/login", {"user": "alex", "password": OLD_PASSWORD})[
    0
]
assert old_access == 401 and fresh_login == 403
print("CONTAINED: old session", old_access, "; fresh login", fresh_login)
print(
    "Verified by rejected requests, not merely by an admin success message."
)


In [ ]:
account["password_hash"] = password_hash(NEW_PASSWORD)
account["enabled"] = True
record("admin-reset-and-enable", "completed")
assert (
    request("/login", {"user": "alex", "password": OLD_PASSWORD})[0] == 401
)
assert request("/order", token=old_token)[0] == 401
status, body = request("/login", {"user": "alex", "password": NEW_PASSWORD})
assert status == 200 and request("/order", token=body["token"])[0] == 200
print(
    "RECOVERY PASS: old password 401; old token 401; new login and order access 200."
)
Path("project1_response_audit.json").write_text(json.dumps(audit, indent=2))
server.shutdown()
server.server_close()
print("Training server stopped. No enterprise accounts were changed.")
